In [1]:
from langchain_core.documents import Document
import pandas as pd
from langchain_community.document_loaders import PyPDFLoader, DataFrameLoader
import logging
import sqlite3
from groq import Groq

C:\Users\User\AppData\Local\Temp\ipykernel_768\1739051761.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, DataFrameLoader
c:\github\Telecom_rag\rag_tele\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os
from dotenv import load_dotenv
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

In [ ]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import DataFrameLoader
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
import pandas as pd
import sqlite3
import os
import logging

class telecom():
    def __init__(self):
        logging.info("Initializing the telecom class")
        
    def build_rag(self, vectorstore):
        # Groq model (LLM)
        llm = ChatGroq(
            model="openai/gpt-oss-20b",
            groq_api_key=os.getenv("GROQ_API_KEY"),
            temperature=0
        )

        # Prompt template
        system_prompt = (
            "You are a helpful telecom support assistant. "
            "Use retrieved context to answer. When unsure, say \"I don't have enough information\"."
        )

        prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            ("human", "Context:\n{context}\n\nQuestion: {input}")
        ])

        retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
        
        # Simple RAG chain using pipe operator
        rag_chain = prompt | llm | StrOutputParser()
        
        def answer_question(question):
            # context_docs = retriever.get_relevant_documents(question)
            context_docs = retriever.invoke(question)
            context = "\n".join([doc.page_content for doc in context_docs])
            return rag_chain.invoke({"context": context, "input": question})
        
        return answer_question

    def load_all_documents(self,
        pdf_path: str = "data/telecom_guide.pdf",
        excel_path: str = "data/faq.csv",
        db_path: str = "data/tickets.db"
    ):
        docs = []

        # 1. PDF
        pdf_loader = PyPDFLoader(pdf_path)
        pdf_docs = pdf_loader.load()
        docs.extend(pdf_docs)
        print(f"✅ Loaded {len(pdf_docs)} pages from PDF")

        # 2. Excel (FAQ)
        df = pd.read_csv(excel_path)
        df_loader = DataFrameLoader(df, page_content_column="answer")
        excel_docs = df_loader.load()
        docs.extend(excel_docs)
        print(f"✅ Loaded {len(excel_docs)} FAQs from Excel")

        # 3. SQLite (Tickets)
        conn = sqlite3.connect(db_path)
        df_tickets = pd.read_sql_query("SELECT * FROM tickets", conn)
        conn.close()

        for _, row in df_tickets.iterrows():
            content = " | ".join([f"{col}: {val}" for col, val in row.items()])
            docs.append(Document(page_content=content, metadata={"source": "tickets.db"}))
        print(f"✅ Loaded {len(df_tickets)} tickets from DB")

        return docs

In [4]:

# def main():

#     obj = telecom()
#     # logging.info(docs)
#     print("🔄 Loading documents...")
#     docs = obj.load_all_documents()

#     print("🔄 Splitting into chunks...")
#     splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
#     chunks = splitter.split_documents(docs)
#     print(f"✅ Created {len(chunks)} chunks")

#     print("🔄 Creating embeddings & vector store (Chroma)...")
#     embeddings = GoogleGenerativeAIEmbeddings(
#         model="gemini-embedding-2-preview",
#         google_api_key=os.getenv("GEMINI_API_KEY")
#     )

#     vectorstore = Chroma.from_documents(
#         documents=chunks,
#         embedding=embeddings,
#         persist_directory="chroma_db"
#     )
#     print("✅ Vector store ready!")

#     rag_chain = obj.build_rag(vectorstore)

#     print("\n✅ RAG system ready! Type 'quit' to exit.\n")
#     while True:
#         question = input("You: ")
#         if question.lower() in ["quit", "exit"]:
#             break
#         response = rag_chain.invoke({"input": question})
#         print(f"Assistant: {response['answer']}\n")

In [5]:
import shutil
import os

# Delete old vectorstore
if os.path.exists("chroma_db"):
    shutil.rmtree("chroma_db")
    print("✅ Deleted old chroma_db")

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

if __name__ == "__main__":
    obj = telecom()
    
    # Load documents
    docs = obj.load_all_documents()
    
    # Split documents
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    split_docs = text_splitter.split_documents(docs)
    
    # Create vectorstore with consistent embeddings
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vectorstore = Chroma.from_documents(
        documents=split_docs,
        embedding=embeddings,
        persist_directory="chroma_db"
    )
    print("✅ Vector store ready!")
    
    # Build RAG chain
    rag_chain = obj.build_rag(vectorstore)
    print("✅ RAG system ready! Type 'quit' to exit.\n")
    
    # Interactive loop
    while True:
        query = input("You: ")
        if query.lower() == 'quit':
            break
        
        response = rag_chain(query)
        print(f"\nAssistant: {response}\n")

✅ Deleted old chroma_db
✅ Loaded 9 pages from PDF
✅ Loaded 25 FAQs from Excel
✅ Loaded 20 tickets from DB


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1629.98it/s]


✅ Vector store ready!
✅ RAG system ready! Type 'quit' to exit.


Assistant: It sounds like your internet connection isn’t working. Let’s try a few quick steps to see if we can get it back online:

1. **Check the Network Status**  
   - Open the **MyTelecom** app → **Help** → **Network Status** and enter your postcode.  
   - Or visit the live status page: <https://telecom.example.com/status>  
   - If there’s a listed outage or maintenance window, that’s likely the cause.

2. **Basic Phone Troubleshooting**  
   - **Power cycle**: Turn your phone off, wait 10 seconds, then turn it back on.  
   - **SIM card**: Remove the SIM card, clean the contacts, reseat it firmly, and power on again.  
   - If you have another phone, try inserting the SIM there to rule out a SIM issue.

3. **If the problem persists**  
   - **Report the issue**:  
     - Call **611** (or your local support number).  
     - Or use the MyTelecom app → **Help** → **Report an Issue**.  
     - You can also tweet **@Te